In [ ]:
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#function adapted from the cell2location tutorial 
# Kleshchevnikov, V., Shmatko, A., Dann, E. et al. Cell2location maps fine-grained cell types in spatial transcriptomics. Nat Biotechnol (2022).
def read_and_qc(sample_name, path):
    r""" This function reads the data for one 10X spatial experiment into the anndata object.
    It also calculates QC metrics. Modify this function if required by your workflow.

    :param sample_name: Name of the sample
    :param path: path to data
    """

    adata = sc.read_visium(os.path.join(path, str(sample_name), 'outs'),
                           count_file='filtered_feature_bc_matrix.h5', load_images=True)
    adata.obs['sample'] = sample_name
    adata.var['SYMBOL'] = adata.var_names
    adata.var.drop(columns='gene_ids', inplace=True)
    adata.var_names_make_unique()
    
    # Calculate QC metrics
    from scipy.sparse import csr_matrix
    adata.X = adata.X.toarray()
    sc.pp.calculate_qc_metrics(adata, inplace=True)
    adata.X = csr_matrix(adata.X)
    #also account for genes with MTRNR
    adata.var['mt'] = [gene.startswith('MT') for gene in adata.var['SYMBOL']]
    adata.obs['mt_frac'] = adata[:, adata.var['mt'].tolist()].X.sum(1).A.squeeze()/adata.obs['total_counts']
    #detect ribosomal genes
    adata.var['rp'] = [gene.startswith('RP') for gene in adata.var['SYMBOL']]
  
    # add sample name to obs names
    adata.obs["sample"] = [str(i) for i in adata.obs['sample']]
    adata.obs_names = adata.obs["sample"] + '_' + adata.obs_names
    adata.obs.index.name = 'spot_id'

    return adata
def select_slide(adata, s, s_col='sample'):
    r""" This function selects the data for one slide from the spatial anndata object.

    :param adata: Anndata object with multiple spatial experiments
    :param s: name of selected experiment
    :param s_col: column in adata.obs listing experiment name for each location
    """

    slide = adata[adata.obs[s_col].isin([s]), :]
    s_keys = list(slide.uns['spatial'].keys())
    s_spatial = np.array(s_keys)[[s in k for k in s_keys]][0]

    slide.uns['spatial'] = {s_spatial: slide.uns['spatial'][s_spatial]}

    return slide

In [ ]:
# Read the list of spatial experiments. We renamed the tissue_positions.csv to tissue_positions_list.csv in /spatial
sample_data = pd.read_csv('/g/korbel/olisov/cell2/linda/samples11.csv', names=['Sample'])
# Read the data into anndata objects
slides = []
for i in sample_data['Sample']:
    slides.append(read_and_qc(i, path='/g/korbel/olisov/Visium_Linda/Visium_Data_Linda'))

In [ ]:
adata = slides[0].concatenate(
    slides[1:],
    batch_key="sample",
    uns_merge="unique",
    batch_categories=sample_data['Sample'],
    index_unique=None)
#exclude MT genes
adata.obsm['mt'] = adata[:, adata.var['mt'].values].X.toarray()
adata = adata[:, ~adata.var['mt'].values]

In [ ]:
adata.obs

## QC

In [ ]:
#add barcode column
index_series = adata.obs.index.to_series()
# Split the index by '_' and extract relevant parts
split_parts = index_series.str.split('_', expand=True)
# Combine LM ID with suffix after 'count_'
barcode = split_parts[1] + '_' + split_parts[3]
# Remove 'lane1' if present in the resulting string
adata.obs['Barcode'] = barcode.str.replace('lane1', '', regex=False)

In [ ]:
#calculate qc metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
ax=sc.pl.scatter(adata, x='total_counts', y='n_genes_by_counts', show=False)
ax.set_ylim(0,10000)

In [ ]:
#distribution of counts by spots
plt.hist(adata.obs['total_counts'], bins = 50, color='skyblue', edgecolor='black')
plt.xlabel('number of counts')
plt.ylabel('number of spots')
plt.show()

In [ ]:
np.quantile(adata.obs['total_counts'], 0.99)

In [ ]:
#remove spots above 99% of counts
adata = adata[adata.obs['total_counts'] <= 18346].copy()

In [ ]:
#distribution of genes by spots
plt.hist(adata.obs['n_genes_by_counts'], bins = 50, color='skyblue', edgecolor='black')
plt.xlabel('number of genes')
plt.ylabel('number of spots')
plt.show()

In [ ]:
#remove spots wih too few genes
original_cells = adata.obs_names.copy()
sc.settings.verbosity = 3
sc.pp.filter_cells(adata, min_genes=200)

In [ ]:
#modify sample names
def extract_part(s):
    try:
        start = s.find('e1') + 2  # Find the position after 'e1'
        end = s.rfind('_')       # Find the position of the last '_'
        if start > 0 and end > start:
            return s[start:end]  # Extract the part between the positions
        else:
            return s  # Return original string if conditions are not met
    except Exception as e:
        return s  # Return original string in case of any exception

# Apply the function to the sample column
adata.obs['sample'] = adata.obs['sample'].apply(extract_part)

In [ ]:
adata.obs['sample'] = adata.obs['sample'].str.replace('B', '')

In [ ]:
#modify the barcode column sample_umi
adata.obs['Barcode'] = adata.obs['sample'].astype(str) + '_' + adata.obs['Barcode'].apply(lambda x: x.split('_')[-1])

In [ ]:
adata.obs

In [ ]:
#save the resulting raw filtered adata object
adata.write("/g/korbel/olisov/Visium_Linda/adata_objects/11samples_raw.h5ad", compression="gzip")

In [ ]:
#plot number of transcripts and genes per spot per sample
samples = adata.obs['sample'].unique()
colors = plt.cm.tab20(range(len(samples)))
color_map = dict(zip(samples, colors))
hex_colors = [mcolors.to_hex(color_map[sample]) for sample in samples]
palette = dict(zip(samples, hex_colors))


# Figure dimensions
fig_width_mm = 43.5
fig_width_inch = fig_width_mm / 25.4
fig_height_inch = fig_width_inch * 0.8

# Plot
fig, ax = plt.subplots(figsize=(fig_width_inch, fig_height_inch), dpi=600)
sns.violinplot(
    data=adata.obs,
    x='sample',
    y='n_genes_by_counts',
    order=samples,
    palette=palette,
    scale='width',
    inner='quart',
    cut=0,
    linewidth=0.3,
    width=0.8,
    ax=ax
)

# Style the plot
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_linewidth(0.3)
ax.spines['bottom'].set_linewidth(0.3)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right',
    rotation_mode='anchor',
    fontsize=5)

ax.tick_params(width=0.3, labelsize=5)
plt.xticks(rotation=45, ha='right')
plt.xlabel('')
plt.ylabel(f'Unique genes per spot', fontsize=5)

plt.tight_layout()
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/totalgenes_violin_by_sample.svg', dpi=600)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(fig_width_inch, fig_height_inch), dpi=600)
sns.violinplot(
    data=adata.obs,
    x='sample',
    y='total_counts',
    order=samples,
    palette=palette,
    scale='width',
    inner='quart',
    cut=0,
    linewidth=0.3,
    width=0.8,
    ax=ax
)

# Style the plot
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_linewidth(0.3)
ax.spines['bottom'].set_linewidth(0.3)

ax.set_xticklabels(
    ax.get_xticklabels(),
    rotation=45,
    ha='right',
    rotation_mode='anchor',
    fontsize=5)

ax.tick_params(width=0.3, labelsize=5)
plt.xticks(rotation=45, ha='right')
plt.xlabel('')
plt.ylabel(f'Total counts per spot', fontsize=5)

plt.tight_layout()
plt.savefig(f'/g/korbel/olisov/Visium_Linda/paper_figures/11samples/totalcounts_violin_by_sample.svg', dpi=600)
plt.show()

In [ ]:
# scale each cell to a common library size
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)

In [ ]:
sc.pp.log1p(adata)

In [ ]:
#save the normalized file
adata.write("/g/korbel/olisov/Visium_Linda/adata_objects/11samples_normalized.h5ad", compression="gzip")

In [ ]:
#read the normalized file
adata=anndata.read("/g/korbel/olisov/Visium_Linda/adata_objects/11samples_normalized.h5ad")

# Stiffness data

In [ ]:
#load the values after MAD
filtered_df=pd.read_csv('/g/korbel/olisov/Visium_Linda/stiffness/stiffness_measurements_50um/merged/11_samples_mapping_final.tsv', sep='\t')
filtered_df["Y"] = pd.to_numeric(filtered_df["Y"])
#plot the distribution of stiffness values
plt.hist(filtered_df['Y'])
plt.xlabel('Y')
plt.ylabel('measurement count')
plt.show()

In [ ]:
filtered_df=filtered_df[filtered_df['Y'] <= 12500]

In [ ]:
filtered_df

In [ ]:
# average of the stiffness for a spot, min and max values

# Perform the groupby and aggregate operations
stiff_df = (filtered_df.groupby('Barcode')
            .agg(mean=('Y', 'mean'), 
                 min=('Y', 'min'), 
                 max=('Y', 'max'))
            .reset_index())

# Set Barcode as the index
stiff_df.set_index('Barcode', inplace=True)

In [ ]:
stiff_df

In [ ]:
# Merging the dataframes, adding the columns from df to adata.obs
adata.obs = adata.obs.merge(stiff_df, left_on='Barcode', right_index=True, how='left')
adata = adata[~adata.obs['mean'].isna()].copy()

In [ ]:
#remove genes found in less than 10 spots
sc.pp.filter_genes(adata, min_cells=10)

In [ ]:
#count spots per sample
adata.obs['sample'].value_counts()

In [ ]:
exclude_genes = adata.var['rp'].values
# Filter out RP genes from adata
adataex = adata[:, ~exclude_genes]
#create df with barcodes and genes
adataex.var['SYMBOL']=adataex.var.index
df = pd.DataFrame(adataex.X.toarray(), columns=adataex.var['SYMBOL'], index=adataex.obs['Barcode'])
#add stiffness values
df1=pd.merge(df, stiff_df, how='inner', on='Barcode')

In [ ]:
df1

In [ ]:
df1.to_csv('/g/korbel/olisov/Visium_Linda/stiffness/normalized/merged_11samples_50um_final.tsv', sep='\t')

# Export data for DE analysis

In [ ]:
#Raw counts are used here
adata=anndata.read('/g/korbel/olisov/Visium_Linda/adata_objects/11samples_raw.h5ad')
adata.obs = adata.obs.merge(stiff_df, left_on='Barcode', right_index=True, how='left')
adata = adata[~adata.obs['mean'].isna()].copy()
sc.pp.filter_genes(adata, min_cells=10)
exclude_genes = adata.var['rp'].values
adata = adata[:, ~exclude_genes]

In [ ]:
#identify stiffness thresholds
sns.ecdfplot(adata.obs['mean'], linewidth=2, color='gray')

In [ ]:
#we split the data into halves based on the mean stiffness
adata1=adata.copy()
np.quantile(adata.obs['mean'], 0.5)

In [ ]:
condition_ecm_low = adata1.obs['mean'] <= 275
condition_ecm_high = adata1.obs['mean'] > 275

# Apply the conditions to create the new column
adata1.obs['Barcode_DE'] = np.where(condition_ecm_low, 'ECM_low', np.where(condition_ecm_high, 'ECM_high', 'none'))

# Verify the new column
print(adata1.obs['Barcode_DE'].value_counts())

#remove NaNs
adata1 = adata1[adata1.obs['Barcode_DE']!='none'].copy()

#save the data for analysis in R:
# Save the counts matrix (assay)
counts_df = pd.DataFrame(adata1.X.T.todense(), index=adata1.var_names, columns=adata1.obs_names)
counts_df.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_counts_275.csv")

# Save metadata (colData)
adata1.obs.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_coldata_275.csv")

# Save variable metadata if needed (rowData)
adata1.var.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_rowdata_275.csv")

In [ ]:
#twice the q50 stiffness
condition_ecm_low = adata1.obs['mean'] <= 550
condition_ecm_high = adata1.obs['mean'] > 550

# Apply the conditions to create the new column
adata1.obs['Barcode_DE'] = np.where(condition_ecm_low, 'ECM_low', np.where(condition_ecm_high, 'ECM_high', 'none'))

# Verify the new column
print(adata1.obs['Barcode_DE'].value_counts())

#remove NaNs
adata1 = adata1[adata1.obs['Barcode_DE']!='none'].copy()

#save the data for analysis in R:
# Save the counts matrix (assay)
counts_df = pd.DataFrame(adata1.X.T.todense(), index=adata1.var_names, columns=adata1.obs_names)
counts_df.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_counts_550.csv")

# Save metadata (colData)
adata1.obs.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_coldata_550.csv")

# Save variable metadata if needed (rowData)
adata1.var.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_rowdata_550.csv")

In [ ]:
#thrice the q50 
#twice the q50 stiffness
condition_ecm_low = adata1.obs['mean'] <= 825
condition_ecm_high = adata1.obs['mean'] > 825

# Apply the conditions to create the new column
adata1.obs['Barcode_DE'] = np.where(condition_ecm_low, 'ECM_low', np.where(condition_ecm_high, 'ECM_high', 'none'))

# Verify the new column
print(adata1.obs['Barcode_DE'].value_counts())

#remove NaNs
adata1 = adata1[adata1.obs['Barcode_DE']!='none'].copy()

#save the data for analysis in R:
# Save the counts matrix (assay)
counts_df = pd.DataFrame(adata1.X.T.todense(), index=adata1.var_names, columns=adata1.obs_names)
counts_df.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_counts_825.csv")

# Save metadata (colData)
adata1.obs.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_coldata_825.csv")

# Save variable metadata if needed (rowData)
adata1.var.to_csv("/g/korbel/olisov/Visium_Linda/DE/11_rowdata_825.csv")